# Real 2024 Vecchia: GPU-Batched Point-Target Hybrid

This notebook fits one selected point-target hybrid Vecchia model.

Two presets are kept here:

- `cluster_matched`: point-target comparison for the new 3x3 cluster model. Conditioning is `A6; B=same+local1+fresh1; C=same+fresh1`.
- `lean_41`: previous stronger hybrid baseline `A20; B=same+local8+fresh4; C=same+local4+fresh3`.

For a clean block-vs-point comparison, start with `MODEL_PRESET = "cluster_matched"`. The likelihood uses actual source coordinates through `keep_ori=True`; the regular grid ordering/NNS map is only the shared point-neighbor scaffold.

In [12]:
import os
import sys
import time
import io
import contextlib
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.nn import Parameter

AMAREL_SRC = "/home/jl2815/tco"
LOCAL_SRC = "/Users/joonwonlee/Documents/GEMS_TCO-1/src"
_src = AMAREL_SRC if os.path.exists(AMAREL_SRC) else LOCAL_SRC
sys.path.insert(0, _src)

from GEMS_TCO import configuration as config
from GEMS_TCO.data_loader import load_data_dynamic_processed
from GEMS_TCO.kernels_vecchia_hybrid import HybridVecchiaFit

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float64

print("SRC:", _src)
print("DEVICE:", DEVICE)


ROUND_DECIMALS = 4

def round_numeric_df(df, digits=ROUND_DECIMALS):
    out = df.copy()
    num_cols = out.select_dtypes(include=[np.number]).columns
    out[num_cols] = out[num_cols].round(digits)
    return out

def round_numeric_series(s, digits=ROUND_DECIMALS):
    out = s.copy()
    for idx, val in out.items():
        if isinstance(val, (float, np.floating)) and np.isfinite(val):
            out[idx] = round(float(val), digits)
    return out


SRC: /Users/joonwonlee/Documents/GEMS_TCO-1/src
DEVICE: cpu


## Settings

`keep_ori=True` restores the real source locations for the Vecchia likelihood. The regular grid is used for shared ordering and the base neighbor scaffold only.

In [ ]:
YEAR = "2024"
MONTH = 7
DAYS_LIST = [13]  # 0-based: [13] is July 14. For full July use list(range(31)).
LAT_RANGE = [-3, 2]
LON_RANGE = [121, 131]

LAT_LON_RESOLUTION = [1, 1]
DELTA_LAT_BASE = 0.044
DELTA_LON_BASE = 0.063
MM_COND_NUMBER = 100  # 30 is enough for cluster_matched; 100 keeps lean_41 available.
SMOOTH = 0.5
NHEADS = 0
DAILY_STRIDE = 2

# For block-vs-point comparison use cluster_matched. For the old stronger baseline use lean_41.

#MODEL_PRESET = "cluster_matched"  # cluster_matched or lean_41
MODEL_PRESET = "lean_41"  

LBFGS_LR = 1.0
LBFGS_MAX_STEPS = 20
LBFGS_MAX_EVAL = 20
LBFGS_HISTORY_SIZE = 10
GRAD_TOL = 1e-5
SUPPRESS_FIT_PRINTS = True

# Godambe is expensive. Turn on after the basic fit looks stable.
COMPUTE_GODAMBE = False
HESSIAN_EPS = 1e-4
SCORE_EPS = 1e-5
H_RIDGE_SCALE = 1e-6
GODAMBE_J_METHOD = "block"  # block, per_unit_centered, per_unit_uncentered
GODAMBE_BLOCK_LAT_WIDTH = 0.50
GODAMBE_BLOCK_LON_WIDTH = 0.50
GODAMBE_BLOCK_TIME_WIDTH = 2.0

OUT_DIR = Path("/Users/joonwonlee/Documents/GEMS_TCO-1/Exercises/st_model/day/local_computer/log")
OUT_PREFIX = "real_vecc_2024_point_hybrid_051326"
SAVE_CSV = True

INIT_PHYSICAL = {
    "sigmasq": 10.0,
    "range_lat": 0.30,
    "range_lon": 0.40,
    "range_time": 2.0,
    "advec_lat": 0.05,
    "advec_lon": -0.10,
    "nugget": 2.5,
}

print("days:", DAYS_LIST)
print("smooth:", SMOOTH)
print("mm_cond_number:", MM_COND_NUMBER)
print("model preset:", MODEL_PRESET)
print("Godambe:", COMPUTE_GODAMBE, GODAMBE_J_METHOD)

days: [13]
smooth: 0.5
mm_cond_number: 100
model preset: lean_41
Godambe: False block


## Model Specs

In [14]:
def offset_tag(x):
    sign = "p" if x >= 0 else "m"
    return sign + f"{abs(float(x)):.3f}".replace(".", "p")


def hybrid_cluster_matched_spec(offset=0.063):
    tag = offset_tag(offset)
    return {
        "name": f"PointHybrid_MatchCluster_A06_BsL01F01_CsF01_O{tag}",
        "group": "point_hybrid_cluster_matched",
        "kernel": "hybrid_fresh",
        "limit_A": 6,
        "limit_B": 1,
        "limit_C": 0,
        "lag1_local_count": 1,
        "lag1_fresh_count": 1,
        "lag2_local_count": 0,
        "lag2_fresh_count": 1,
        "pred_lag1_lon_offset": float(offset),
        "pred_lag2_lon_offset": float(2.0 * offset),
        "pred_lag1_cells": float(offset / DELTA_LON_BASE),
        "pred_lag2_cells": float(2.0 * offset / DELTA_LON_BASE),
        "total_conditioning": 6 + 1 + 1 + 1 + 1 + 0 + 1,
        "allocation": f"A6; B=same+local1+fresh1; C=same+fresh1; lag1 offset={offset:.4f}",
        "description": "Point-target hybrid with the same block-count allocation as cluster_hybrid default",
    }


def hybrid_lean_spec(offset=0.063):
    tag = offset_tag(offset)
    return {
        "name": f"Hybrid_Lean_L08F04_C4F03_O{tag}",
        "group": "hybrid_lean",
        "kernel": "hybrid_fresh",
        "limit_A": 20,
        "limit_B": 8,
        "limit_C": 4,
        "lag1_local_count": 8,
        "lag1_fresh_count": 4,
        "lag2_local_count": 4,
        "lag2_fresh_count": 3,
        "pred_lag1_lon_offset": float(offset),
        "pred_lag2_lon_offset": float(2.0 * offset),
        "pred_lag1_cells": float(offset / DELTA_LON_BASE),
        "pred_lag2_cells": float(2.0 * offset / DELTA_LON_BASE),
        "total_conditioning": 20 + 1 + 8 + 4 + 1 + 4 + 3,
        "allocation": f"A20; B=same+local8+fresh4; C=same+local4+fresh3; lag1 offset={offset:.4f}",
        "description": "Previous GPU-batched hybrid lean baseline",
    }


PRESET_BUILDERS = {
    "cluster_matched": hybrid_cluster_matched_spec,
    "lean_41": hybrid_lean_spec,
}
if MODEL_PRESET not in PRESET_BUILDERS:
    raise ValueError(f"Unknown MODEL_PRESET={MODEL_PRESET}; choose one of {sorted(PRESET_BUILDERS)}")

MODEL_SPEC = PRESET_BUILDERS[MODEL_PRESET](0.063)
MODEL_KEY = f"{MODEL_PRESET}::{MODEL_SPEC['name']}"
MODEL_SAVE_TAG = MODEL_SPEC["name"].replace(".", "p").replace("-", "m")

spec_df = pd.DataFrame([MODEL_SPEC], index=[MODEL_KEY])
print("model:", MODEL_SPEC["name"])
print("fits:", len(DAYS_LIST))
display(round_numeric_df(spec_df[["group", "kernel", "limit_A", "lag1_local_count", "lag1_fresh_count", "lag2_local_count", "lag2_fresh_count", "pred_lag1_lon_offset", "total_conditioning", "allocation"]]))

model: Hybrid_Lean_L08F04_C4F03_Op0p063
fits: 1


,group,kernel,limit_A,lag1_local_count,lag1_fresh_count,lag2_local_count,lag2_fresh_count,pred_lag1_lon_offset,total_conditioning,allocation
lean_41::Hybrid_Lean_L08F04_C4F03_Op0p063,hybrid_lean,hybrid_fresh,20,8,4,4,3,0.063,41,A20; B=same+local8+fresh4; C=same+local4+fresh...


## Load 2024 July Real Data

`is_whittle=False` generates the shared max-min ordering/NNS map for Vecchia.

`keep_ori=True` restores the real source locations for the likelihood. The regular grid ordering is used only as the common ordering and neighbor scaffold.


In [15]:
data_load_instance = load_data_dynamic_processed(config.mac_data_load_path)

df_map, ord_mm, nns_map, monthly_mean = data_load_instance.load_maxmin_ordered_data_bymonthyear(
    lat_lon_resolution=LAT_LON_RESOLUTION,
    mm_cond_number=MM_COND_NUMBER,
    years_=[YEAR],
    months_=[MONTH],
    lat_range=LAT_RANGE,
    lon_range=LON_RANGE,
    is_whittle=False,
)

if ord_mm is None or nns_map is None:
    raise RuntimeError("Vecchia ordering failed: ord_mm/nns_map is None")

key_idx = sorted(df_map)
print("n hourly slots:", len(key_idx))
print("monthly_mean:", monthly_mean)
print("ord length:", len(ord_mm), "nns shape/len:", getattr(nns_map, "shape", len(nns_map)))
print("first key:", key_idx[0], "last key:", key_idx[-1])

first_df_ord = df_map[key_idx[0]].iloc[ord_mm].reset_index(drop=True)
ordered_base_coords_np = first_df_ord[["Latitude", "Longitude"]].to_numpy(dtype=np.float64)
print("ordered_base_coords_np:", ordered_base_coords_np.shape)

--- Global Monthly Mean for 2024-7: 257.9726 ---
--- Generating NNS Map for Vecchia (C++ Accelerated) ---
n hourly slots: 248
monthly_mean: 257.9726104252314
ord length: 18126 nns shape/len: (18126, 100)
first key: 2024_07_y24m07day01_hm00:53 last key: 2024_07_y24m07day31_hm07:48
ordered_base_coords_np: (18126, 2)


In [16]:
def assert_ordered_grid_consistent(keys, ord_mm, base_coords):
    for k in keys:
        coords = df_map[k].iloc[ord_mm].reset_index(drop=True)[["Latitude", "Longitude"]].to_numpy(dtype=np.float64)
        if coords.shape != base_coords.shape or not np.allclose(coords, base_coords, equal_nan=True):
            raise RuntimeError(f"Ordered grid coordinate mismatch at {k}; spatial_coords/nns_map scaffold is not reusable.")

for day_idx in DAYS_LIST:
    hour_indices = [day_idx * 8, (day_idx + 1) * 8]
    assert_ordered_grid_consistent(key_idx[hour_indices[0]:hour_indices[1]], ord_mm, ordered_base_coords_np)
print("ordered grid consistency check passed for selected days")

ordered grid consistency check passed for selected days


In [17]:
daily_hourly_maps_vecc = {}
daily_aggregated_tensors_vecc = {}
selected_key_map = {}

for day_idx in DAYS_LIST:
    hour_indices = [day_idx * 8, (day_idx + 1) * 8]
    selected_key_map[day_idx] = key_idx[hour_indices[0]:hour_indices[1]]
    day_hourly_map, day_aggregated_tensor = data_load_instance.load_working_data(
        df_map,
        monthly_mean,
        hour_indices,
        ord_mm=ord_mm,
        dtype=DTYPE,
        keep_ori=True,
    )
    daily_hourly_maps_vecc[day_idx] = day_hourly_map
    daily_aggregated_tensors_vecc[day_idx] = day_aggregated_tensor

load_rows = []
for day_idx in DAYS_LIST:
    maps = daily_hourly_maps_vecc[day_idx]
    n_valid = sum(int((~torch.isnan(v[:, 2])).sum().item()) for v in maps.values())
    n_total = sum(len(v) for v in maps.values())
    load_rows.append({
        "day_idx": day_idx,
        "day": f"{YEAR}-{MONTH:02d}-{day_idx + 1:02d}",
        "n_time_slots": len(maps),
        "n_rows_total": n_total,
        "n_valid_o3": n_valid,
        "valid_rate": n_valid / max(n_total, 1),
        "first_slot": selected_key_map[day_idx][0] if selected_key_map[day_idx] else None,
        "last_slot": selected_key_map[day_idx][-1] if selected_key_map[day_idx] else None,
    })
load_summary = pd.DataFrame(load_rows)
display(round_numeric_df(load_summary))

,day_idx,day,n_time_slots,n_rows_total,n_valid_o3,valid_rate,first_slot,last_slot
0,13,2024-07-14,8,145008,144078,0.993587,2024_07_y24m07day14_hm00:53,2024_07_y24m07day14_hm07:48


## Hybrid Lean Kernel

The model class is imported from:

`GEMS_TCO.kernels_vecchia_hybrid.HybridVecchiaFit`

No hybrid class is defined inside this notebook. That keeps the experiment notebook independent from kernel implementation details.


In [18]:
# The hybrid implementation lives in src/GEMS_TCO/kernels_vecchia_hybrid.py.
# This cell is intentionally empty so the notebook does not carry a private copy
# of the kernel logic.


## Parameter And Godambe Helpers

In [19]:
P_LABELS = ["sigmasq", "range_lat", "range_lon", "range_time", "advec_lat", "advec_lon", "nugget"]


def physical_to_log_phi(params):
    sigmasq = float(params["sigmasq"])
    range_lat = float(params["range_lat"])
    range_lon = float(params["range_lon"])
    range_time = float(params["range_time"])
    nugget = float(params["nugget"])
    phi2 = 1.0 / range_lon
    phi1 = sigmasq * phi2
    phi3 = (range_lon / range_lat) ** 2
    phi4 = (range_lon / range_time) ** 2
    return [
        np.log(phi1), np.log(phi2), np.log(phi3), np.log(phi4),
        float(params["advec_lat"]), float(params["advec_lon"]), np.log(nugget),
    ]


def transform_log_phi_to_physical(p):
    phi1, phi2, phi3, phi4 = (torch.exp(p[i]) for i in range(4))
    rlon = 1.0 / phi2
    return torch.stack([
        phi1 / phi2,
        rlon / torch.sqrt(phi3),
        rlon,
        rlon / torch.sqrt(phi4),
        p[4],
        p[5],
        torch.exp(p[6]),
    ])


def backmap_params(out_params):
    p = [x.item() if isinstance(x, torch.Tensor) else float(x) for x in out_params[:7]]
    phi2 = np.exp(p[1])
    phi3 = np.exp(p[2])
    phi4 = np.exp(p[3])
    rlon = 1.0 / phi2
    return {
        "sigmasq": np.exp(p[0]) / phi2,
        "range_lat": rlon / phi3 ** 0.5,
        "range_lon": rlon,
        "range_time": rlon / phi4 ** 0.5,
        "advec_lat": p[4],
        "advec_lon": p[5],
        "nugget": np.exp(p[6]),
    }


def relative_to_estimate(se_by_key, est_by_key, key, zero_thresh=0.01):
    denom = abs(float(est_by_key[key]))
    if denom >= zero_thresh:
        return float(se_by_key[key] / denom)
    return float(se_by_key[key])


def finite_diff_hessian(nll_fn, p, eps=HESSIAN_EPS):
    n = p.shape[0]
    H = torch.zeros(n, n, device=p.device, dtype=p.dtype)
    for i in range(n):
        p_p = p.detach().clone(); p_m = p.detach().clone()
        p_p[i] += eps; p_m[i] -= eps
        p_p.requires_grad_(True); p_m.requires_grad_(True)
        g_p = torch.autograd.grad(nll_fn(p_p), p_p)[0].detach()
        g_m = torch.autograd.grad(nll_fn(p_m), p_m)[0].detach()
        H[i] = (g_p - g_m) / (2.0 * eps)
    return (H + H.T) / 2.0


def vecchia_per_unit_target_coords(model):
    chunks = []
    if model.Heads_data is not None and model.Heads_data.shape[0] > 0:
        chunks.append(model.Heads_data[:, [0, 1, 3]].to(dtype=DTYPE))
    for X_b in [model.X_A, model.X_AB, model.X_ABC]:
        if X_b is not None and X_b.shape[0] > 0:
            chunks.append(X_b[:, -1, :].to(dtype=DTYPE))
    if not chunks:
        return torch.empty((0, 3), device=DEVICE, dtype=DTYPE)
    return torch.cat(chunks, dim=0)


def make_block_ids(target_coords):
    lat = target_coords[:, 0]
    lon = target_coords[:, 1]
    tim = target_coords[:, 2]
    lat_id = torch.floor((lat - lat.min()) / GODAMBE_BLOCK_LAT_WIDTH).to(torch.long)
    lon_id = torch.floor((lon - lon.min()) / GODAMBE_BLOCK_LON_WIDTH).to(torch.long)
    if GODAMBE_BLOCK_TIME_WIDTH is None or GODAMBE_BLOCK_TIME_WIDTH <= 0:
        time_id = torch.zeros_like(lat_id)
    else:
        time_id = torch.floor((tim - tim.min()) / GODAMBE_BLOCK_TIME_WIDTH).to(torch.long)
    n_lon = int(lon_id.max().item()) + 1 if lon_id.numel() else 1
    n_time = int(time_id.max().item()) + 1 if time_id.numel() else 1
    raw_id = (lat_id * n_lon + lon_id) * n_time + time_id
    _, block_id = torch.unique(raw_id, sorted=True, return_inverse=True)
    return block_id


def score_cov_per_unit_centered(score_mat):
    n_units = score_mat.shape[1]
    score_mean = score_mat.mean(dim=1)
    score_centered = score_mat - score_mean.unsqueeze(1)
    if n_units > 1:
        return score_centered @ score_centered.T / (n_units * (n_units - 1))
    return score_mat @ score_mat.T / max(n_units ** 2, 1)


def score_cov_block_cluster(score_mat, target_coords):
    n_units = score_mat.shape[1]
    scores = score_mat.T.contiguous()
    block_id = make_block_ids(target_coords)
    n_blocks = int(block_id.max().item()) + 1 if block_id.numel() else 0
    block_scores = torch.zeros((n_blocks, scores.shape[1]), device=DEVICE, dtype=DTYPE)
    block_scores.index_add_(0, block_id, scores)
    if n_blocks > 1:
        centered = block_scores - block_scores.mean(dim=0, keepdim=True)
        J = centered.T @ centered * (n_blocks / (n_blocks - 1)) / (n_units ** 2)
    else:
        J = block_scores.T @ block_scores / max(n_units ** 2, 1)
    return J, n_blocks


def compute_vecchia_godambe_real(model, raw_params):
    p_hat = torch.tensor(raw_params[:7], device=DEVICE, dtype=DTYPE, requires_grad=True)

    def nll(p):
        return model.vecchia_batched_likelihood(p)

    H = finite_diff_hessian(nll, p_hat)
    eig = torch.linalg.eigvalsh(H).detach()
    h_abs_min = torch.clamp(torch.min(torch.abs(eig)), min=1e-12)
    h_cond = float((torch.max(torch.abs(eig)) / h_abs_min).detach().cpu())
    beta_hat = model.get_gls_beta(p_hat).detach()

    def per_unit_losses(p):
        return model.vecchia_per_unit_nll_terms(p, beta_hat)

    cols = []
    for k in range(p_hat.shape[0]):
        pp = p_hat.detach().clone(); pm = p_hat.detach().clone()
        pp[k] += SCORE_EPS; pm[k] -= SCORE_EPS
        with torch.no_grad():
            cols.append((per_unit_losses(pp) - per_unit_losses(pm)) / (2.0 * SCORE_EPS))
    score_mat = torch.stack(cols)
    n_units = score_mat.shape[1]
    target_coords = vecchia_per_unit_target_coords(model)
    if target_coords.shape[0] != n_units:
        raise RuntimeError(f"target/score mismatch: targets={target_coords.shape[0]}, scores={n_units}")

    score_mean = score_mat.mean(dim=1)
    p_grad = p_hat.detach().clone().requires_grad_(True)
    profile_grad = torch.autograd.grad(nll(p_grad), p_grad)[0].detach()
    score_grad_diff = profile_grad - score_mean

    J_uncentered = score_mat @ score_mat.T / (n_units ** 2)
    J_centered = score_cov_per_unit_centered(score_mat)
    J_block, n_blocks = score_cov_block_cluster(score_mat, target_coords)
    if GODAMBE_J_METHOD == "block":
        J_main = J_block
    elif GODAMBE_J_METHOD == "per_unit_centered":
        J_main = J_centered
    elif GODAMBE_J_METHOD == "per_unit_uncentered":
        J_main = J_uncentered
    else:
        raise ValueError(f"Unknown GODAMBE_J_METHOD={GODAMBE_J_METHOD!r}")

    eye = torch.eye(H.shape[0], device=DEVICE, dtype=DTYPE)
    h_scale = torch.clamp(torch.mean(torch.abs(torch.diag(H))), min=1.0)
    H_inv = torch.linalg.pinv(H + eye * h_scale * H_RIDGE_SCALE)
    Jac = torch.autograd.functional.jacobian(transform_log_phi_to_physical, p_hat)

    def summarize_J(J):
        G_raw = H_inv @ J @ H_inv
        G_phys = Jac @ G_raw @ Jac.T
        se = torch.sqrt(torch.clamp(torch.diag(G_phys), min=0.0)).detach().cpu().numpy()
        return dict(zip(P_LABELS, [float(x) for x in se]))

    est = backmap_params(raw_params)
    se_main = summarize_J(J_main)
    se_block = summarize_J(J_block)
    se_centered = summarize_J(J_centered)
    se_uncentered = summarize_J(J_uncentered)

    rel_main = {k: relative_to_estimate(se_main, est, k) for k in P_LABELS}
    rel_block = {k: relative_to_estimate(se_block, est, k) for k in P_LABELS}
    rel_centered = {k: relative_to_estimate(se_centered, est, k) for k in P_LABELS}
    rel_uncentered = {k: relative_to_estimate(se_uncentered, est, k) for k in P_LABELS}

    return {
        "gim_j_method": GODAMBE_J_METHOD,
        "gim_n_units": int(n_units),
        "gim_n_blocks": int(n_blocks),
        "gim_h_cond_abs": h_cond,
        "gim_score_mean_max_abs": float(torch.max(torch.abs(score_mean)).detach().cpu()),
        "gim_profile_grad_max_abs": float(torch.max(torch.abs(profile_grad)).detach().cpu()),
        "gim_score_profile_diff_max_abs": float(torch.max(torch.abs(score_grad_diff)).detach().cpu()),
        **{f"gim_se_{k}": v for k, v in se_main.items()},
        **{f"gim_relse_est_{k}": v for k, v in rel_main.items()},
        **{f"gim_relse_est_block_{k}": v for k, v in rel_block.items()},
        **{f"gim_relse_est_centered_{k}": v for k, v in rel_centered.items()},
        **{f"gim_relse_est_uncentered_{k}": v for k, v in rel_uncentered.items()},
    }

## Fit Helpers

In [20]:
def make_params_list(init_physical=INIT_PHYSICAL):
    initial_vals = physical_to_log_phi(init_physical)
    return [Parameter(torch.tensor([val], dtype=DTYPE, device=DEVICE)) for val in initial_vals]


def instantiate_model(spec, daily_hourly_map):
    if spec["kernel"] != "hybrid_fresh":
        raise ValueError(f"unknown kernel {spec['kernel']}")
    return HybridVecchiaFit(
        smooth=SMOOTH,
        input_map=daily_hourly_map,
        nns_map=nns_map,
        mm_cond_number=MM_COND_NUMBER,
        nheads=NHEADS,
        limit_A=spec["limit_A"],
        limit_B_local=spec["lag1_local_count"],
        limit_C_local=spec["lag2_local_count"],
        daily_stride=DAILY_STRIDE,
        spatial_coords=ordered_base_coords_np,
        lag1_lon_offset=spec["pred_lag1_lon_offset"],
        lag1_fresh_count=spec["lag1_fresh_count"],
        lag2_fresh_count=spec["lag2_fresh_count"],
    )


def fit_one(day_idx, model_key, spec):
    daily_hourly_map = daily_hourly_maps_vecc[day_idx]
    params_list = make_params_list()
    model = instantiate_model(spec, daily_hourly_map)

    t0 = time.time()
    if SUPPRESS_FIT_PRINTS:
        with contextlib.redirect_stdout(io.StringIO()):
            model.precompute_conditioning_sets()
    else:
        model.precompute_conditioning_sets()
    precompute_s = time.time() - t0

    optimizer = model.set_optimizer(
        params_list,
        lr=LBFGS_LR,
        max_iter=LBFGS_MAX_EVAL,
        max_eval=LBFGS_MAX_EVAL,
        history_size=LBFGS_HISTORY_SIZE,
    )

    t1 = time.time()
    if SUPPRESS_FIT_PRINTS:
        with contextlib.redirect_stdout(io.StringIO()):
            out, steps_ran = model.fit_vecc_lbfgs(params_list, optimizer, max_steps=LBFGS_MAX_STEPS, grad_tol=GRAD_TOL)
    else:
        out, steps_ran = model.fit_vecc_lbfgs(params_list, optimizer, max_steps=LBFGS_MAX_STEPS, grad_tol=GRAD_TOL)
    fit_s = time.time() - t1

    godambe = {}
    gim_s = 0.0
    if COMPUTE_GODAMBE:
        t2 = time.time()
        godambe = compute_vecchia_godambe_real(model, out[:7])
        gim_s = time.time() - t2

    est = backmap_params(out)
    valid_count = sum(int((~torch.isnan(v[:, 2])).sum().item()) for v in daily_hourly_map.values())
    row = {
        "year": YEAR,
        "month": MONTH,
        "day_idx": int(day_idx),
        "day": f"{YEAR}-{MONTH:02d}-{day_idx + 1:02d}",
        "model_key": model_key,
        "model": spec["name"],
        "group": spec["group"],
        "kernel": spec["kernel"],
        "allocation": spec["allocation"],
        "limit_A": spec["limit_A"],
        "limit_B": spec["limit_B"],
        "limit_C": spec["limit_C"],
        "lag1_local_count": spec["lag1_local_count"],
        "lag1_fresh_count": spec["lag1_fresh_count"],
        "lag2_local_count": spec["lag2_local_count"],
        "lag2_fresh_count": spec["lag2_fresh_count"],
        "pred_lag1_lon_offset": spec.get("pred_lag1_lon_offset", 0.0),
        "pred_lag2_lon_offset": spec.get("pred_lag2_lon_offset", 0.0),
        "pred_lag1_cells": spec.get("pred_lag1_cells", 0.0),
        "pred_lag2_cells": spec.get("pred_lag2_cells", 0.0),
        "total_conditioning": spec["total_conditioning"],
        "daily_stride": DAILY_STRIDE,
        "n_valid_o3": int(valid_count),
        "loss": float(out[-1]),
        "fit_steps": int(steps_ran),
        "precompute_s": round(precompute_s, ROUND_DECIMALS),
        "fit_s": round(fit_s, ROUND_DECIMALS),
        "gim_s": round(gim_s, ROUND_DECIMALS),
        "total_s": round(precompute_s + fit_s + gim_s, ROUND_DECIMALS),
        **{f"{k}_est": float(v) for k, v in est.items()},
    }
    row["range_t_est"] = row.pop("range_time_est")
    row.update(godambe)
    del model, params_list, optimizer
    return row


## Run Hybrid Lean Fit


In [21]:
records = []

for day_idx in DAYS_LIST:
    print(f"{'='*70} Day {day_idx + 1}: {YEAR}-{MONTH:02d}-{day_idx + 1:02d} 1{'='*70}")
    print(f"{MODEL_SPEC['name']}: fitting", end="")
    try:
        row = fit_one(day_idx, MODEL_KEY, MODEL_SPEC)
        records.append(row)
        print(f" | loss={row['loss']:.4f} time={row['total_s']:.4f}s advec_lon={row['advec_lon_est']:+.4f}")
    except Exception as exc:
        import traceback
        print(f" | FAILED: {type(exc).__name__}: {exc}")
        traceback.print_exc()
        records.append({
            "year": YEAR,
            "month": MONTH,
            "day_idx": int(day_idx),
            "day": f"{YEAR}-{MONTH:02d}-{day_idx + 1:02d}",
            "model_key": MODEL_KEY,
            "model": MODEL_SPEC["name"],
            "group": MODEL_SPEC["group"],
            "kernel": MODEL_SPEC["kernel"],
            "error": repr(exc),
        })
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

results_df = pd.DataFrame(records)
if SAVE_CSV:
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    result_path = OUT_DIR / f"{OUT_PREFIX}_{MODEL_SAVE_TAG}_results.csv"
    round_numeric_df(results_df).to_csv(result_path, index=False, float_format="%.4f")
    print("Saved:", result_path)

display(round_numeric_df(results_df))


====================================================================== Day 14: 2024-07-14 1======================================================================
Hybrid_Lean_L08F04_C4F03_Op0p063: fitting | loss=1.196726 time=147.6s advec_lon=-0.0816
Saved: /Users/joonwonlee/Documents/GEMS_TCO-1/Exercises/st_model/day/local_computer/log/real_vecc_2024_point_hybrid_051326_Hybrid_Lean_L08F04_C4F03_Op0p063_results.csv


,year,month,day_idx,day,model_key,model,group,kernel,allocation,limit_A,...,fit_s,gim_s,total_s,sigmasq_est,range_lat_est,range_lon_est,advec_lat_est,advec_lon_est,nugget_est,range_t_est
0,2024,7,13,2024-07-14,lean_41::Hybrid_Lean_L08F04_C4F03_Op0p063,Hybrid_Lean_L08F04_C4F03_Op0p063,hybrid_lean,hybrid_fresh,A20; B=same+local8+fresh4; C=same+local4+fresh...,20,...,141.217,0.0,147.561,8.753058,0.387899,0.475034,0.00365,-0.081617,1.6083,1.861628


## Compare Loss, Parameters, And Godambe SE

In [22]:
if "error" in results_df.columns:
    ok_df = results_df[results_df["error"].isna()].copy() if results_df["error"].notna().any() else results_df.copy()
else:
    ok_df = results_df.copy()

summary_cols = [
    "loss", "total_s", "fit_s", "gim_s",
    "sigmasq_est", "range_lat_est", "range_lon_est", "range_t_est",
    "advec_lat_est", "advec_lon_est", "nugget_est",
]
if COMPUTE_GODAMBE:
    summary_cols += [
        "gim_se_range_lon", "gim_se_range_time", "gim_se_advec_lon", "gim_se_nugget",
        "gim_relse_est_range_lon", "gim_relse_est_range_time", "gim_relse_est_advec_lon", "gim_relse_est_nugget",
        "gim_h_cond_abs", "gim_n_blocks",
    ]
summary_cols = [c for c in summary_cols if c in ok_df.columns]

if ok_df.empty:
    model_summary = pd.DataFrame()
else:
    agg_spec = {f"{c}_mean": (c, "mean") for c in summary_cols}
    agg_spec.update({f"{c}_median": (c, "median") for c in summary_cols})
    model_summary = (
        ok_df
        .groupby(["model", "group", "kernel", "total_conditioning", "pred_lag1_lon_offset"], sort=False)
        .agg(n_days=("day", "nunique"), **agg_spec)
        .reset_index()
    )

print("Single-model summary:")
display(round_numeric_df(model_summary))

if SAVE_CSV:
    summary_path = OUT_DIR / f"{OUT_PREFIX}_{MODEL_SAVE_TAG}_summary.csv"
    round_numeric_df(model_summary).to_csv(summary_path, index=False, float_format="%.4f")
    print("Saved:", summary_path)


Single-model summary:


,model,group,kernel,total_conditioning,pred_lag1_lon_offset,n_days,loss_mean,total_s_mean,fit_s_mean,gim_s_mean,...,total_s_median,fit_s_median,gim_s_median,sigmasq_est_median,range_lat_est_median,range_lon_est_median,range_t_est_median,advec_lat_est_median,advec_lon_est_median,nugget_est_median
0,Hybrid_Lean_L08F04_C4F03_Op0p063,hybrid_lean,hybrid_fresh,41,0.063,1,1.196726,147.561,141.217,0.0,...,147.561,141.217,0.0,8.753058,0.387899,0.475034,1.861628,0.00365,-0.081617,1.6083


Saved: /Users/joonwonlee/Documents/GEMS_TCO-1/Exercises/st_model/day/local_computer/log/real_vecc_2024_point_hybrid_051326_Hybrid_Lean_L08F04_C4F03_Op0p063_summary.csv


## Reading Guide

- This notebook fits one selected point-target hybrid model, controlled by `MODEL_PRESET`.
- For block-vs-point comparison, use `MODEL_PRESET = "cluster_matched"`: A6 at the current hour, lag-1 same-location anchor plus local1/fresh1, and lag-2 same-location anchor plus fresh1.
- The older stronger baseline is `MODEL_PRESET = "lean_41"`: A20 at the current hour, lag-1 same-location anchor plus local8/fresh4, and lag-2 same-location anchor plus local4/fresh3.
- `pred_lag1_lon_offset = 0.063`; lag 2 uses `0.126`.
- The model class lives in `src/GEMS_TCO/kernels_vecchia_hybrid.py`; this notebook only configures and runs it.
- `COMPUTE_GODAMBE=False` by default for speed. Turn it on after the fit itself is stable.